In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# fedavg
# df = pd.read_csv('/home1/liyang/GitHub/Mirage/Mirage/saved_logs/_Aug.04_23.22.30_fedavg/backdoor_tracking_log_nodefense.csv')
# flame
# df = pd.read_csv('/home1/liyang/GitHub/Mirage/Mirage/saved_logs/_Aug.04_23.24.42_flame/backdoor_tracking_log_nodefense.csv')
# krum
df = pd.read_csv('/home1/liyang/GitHub/Mirage/Mirage/saved_logs/_Aug.04_23.25.13_krum/backdoor_tracking_log_nodefense.csv')
# normbound
# df = pd.read_csv('/home1/liyang/GitHub/Mirage/Mirage/saved_logs/_Aug.04_23.25.29_normbound/backdoor_tracking_log_nodefense.csv')

# Create the plot
plt.figure(figsize=(12, 6))

# Plot each ASR column
plt.plot(df['iteration'], df['R1_ASR'], label='R1_ASR', marker='o', markersize=3)
plt.plot(df['iteration'], df['R2_ASR'], label='R2_ASR', marker='o', markersize=3)
plt.plot(df['iteration'], df['R3_ASR'], label='R3_ASR', marker='o', markersize=3)
plt.plot(df['iteration'], df['R4_ASR'], label='R4_ASR', marker='o', markersize=3)

# Add labels and title
plt.xlabel('Iteration')
plt.ylabel('ASR Value')
plt.title('ASR Values Across Iterations')
plt.legend()

# Add grid
plt.grid(True, linestyle='--', alpha=0.7)

# Show the plot
plt.tight_layout()
plt.show()

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re
from pathlib import Path

def analyze_asr_increase(file_path, output_dir='plots'):
    # Convert output_dir to Path object and create directory if needed
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Load CSV
    df = pd.read_csv(file_path)

    # Region color map (from image)
    region_colors = {
        'R1': '#DAA520',  # Yellow
        'R2': '#EA6B66',  # Red
        'R3': '#2E8B57',  # Green
        'R4': '#1E90FF'   # Blue
    }

    # Parse aggregation rule from path
    match = re.search(r'/([^/]+)/backdoor_tracking_log', file_path)
    aggregation_rule = match.group(1) if match else "Unknown"

    # Determine selection method
    selected_counts = df[[f'R{i}_selected' for i in range(1, 5)]].sum(axis=1)
    if all(selected_counts == 1):
        selection_method = "Single Region Selection"
    elif all(selected_counts == 2):
        selection_method = "Two Region Selection"
    else:
        selection_method = "Mixed Region Selection"

    # Collect |Δ_ASR| and iterations
    all_deltas = []
    for t in range(1, len(df)):
        for i in range(1, 5):
            region = f'R{i}'
            if df.loc[t, f'{region}_selected'] == 1:
                delta = abs(df.loc[t, f'{region}_ASR'] - df.loc[t - 1, f'{region}_ASR'])
                all_deltas.append({
                    'Region': region,
                    'Iteration': df.loc[t, 'iteration'],
                    'Delta_ASR': delta
                })

    delta_df = pd.DataFrame(all_deltas)

    # Ensure output_dir is a string path (in case it's a Path object)
    output_dir = str(output_dir)

    # === PLOT 1: |Δ_ASR| over iterations ===
    plt.figure(figsize=(10, 5))
    sns.lineplot(
        data=delta_df,
        x='Iteration',
        y='Delta_ASR',
        hue='Region',
        hue_order=['R1', 'R2', 'R3', 'R4'],
        palette=region_colors,
        marker='o'
    )
    plt.title(f'|Δ_ASR| Over Iterations\n[{aggregation_rule} | {selection_method}]')
    plt.xlabel('Iteration')
    plt.ylabel('|Δ_ASR|')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f"{output_dir}/{aggregation_rule}_delta_asr_over_iterations.png")
    plt.close()

    # === PLOT 2: Accuracy over iterations ===
    plt.figure(figsize=(10, 5))
    sns.lineplot(data=df, x='iteration', y='acc', color='black', marker='o')
    plt.title(f'Accuracy Over Iterations\n[{aggregation_rule} | {selection_method}]')
    plt.xlabel('Iteration')
    plt.ylabel('Accuracy')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f"{output_dir}/{aggregation_rule}_accuracy_over_iterations.png")
    plt.close()

    # === PLOT 3: Distribution of |Δ_ASR| per Region ===
    plt.figure(figsize=(8, 5))
    sns.boxplot(
        data=delta_df,
        x='Region',
        y='Delta_ASR',
        order=['R1', 'R2', 'R3', 'R4'],
        palette=region_colors
    )
    plt.title(f'Distribution of |Δ_ASR| by Region\n[{aggregation_rule} | {selection_method}]')
    plt.xlabel('Region')
    plt.ylabel('|Δ_ASR|')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f"{output_dir}/{aggregation_rule}_delta_asr_distribution_by_region.png")
    plt.close()



In [2]:
# single region selection
# analyze_asr_increase("/home2/liyang/GitHub/Mirage/Mirage/saved_logs/_Aug.05_19.42.58_fedavg/backdoor_tracking_log_nodefense.csv")
# analyze_asr_increase("/home2/liyang/GitHub/Mirage/Mirage/saved_logs/_Aug.05_19.45.30_flame/backdoor_tracking_log_nodefense.csv")
# analyze_asr_increase("/home2/liyang/GitHub/Mirage/Mirage/saved_logs/_Aug.05_19.45.36_krum/backdoor_tracking_log_nodefense.csv")
# analyze_asr_increase("/home2/liyang/GitHub/Mirage/Mirage/saved_logs/_Aug.05_19.45.42_normbound/backdoor_tracking_log_nodefense.csv")
# analyze_asr_increase("/home2/liyang/GitHub/Mirage/Mirage/saved_logs/_Aug.05_21.26.16_geo/backdoor_tracking_log_nodefense.csv", output_dir="plots")
analyze_asr_increase("/home2/liyang/GitHub/Mirage/Mirage/saved_logs/_Aug.11_19.59.34_fedavg/backdoor_tracking_log_nodefense.csv")

/tmp/ipykernel_453649/812404567.py:87: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(


In [ ]:
# double region selection
# analyze_asr_increase("/home2/liyang/GitHub/Mirage/Mirage/saved_logs/_Aug.05_21.26.59_geo/backdoor_tracking_log_nodefense.csv")
# analyze_asr_increase("/home2/liyang/GitHub/Mirage/Mirage/saved_logs/_Aug.06_20.09.11_fedavg/backdoor_tracking_log_nodefense.csv")
# analyze_asr_increase("/home2/liyang/GitHub/Mirage/Mirage/saved_logs/_Aug.06_20.09.11_flame/backdoor_tracking_log_nodefense.csv")
# analyze_asr_increase("/home2/liyang/GitHub/Mirage/Mirage/saved_logs/_Aug.06_20.09.21_krum/backdoor_tracking_log_nodefense.csv")
# analyze_asr_increase("/home2/liyang/GitHub/Mirage/Mirage/saved_logs/_Aug.06_20.09.51_median/backdoor_tracking_log_nodefense.csv")
# analyze_asr_increase("/home2/liyang/GitHub/Mirage/Mirage/saved_logs/_Aug.06_20.10.05_normbound/backdoor_tracking_log_nodefense.csv")
analyze_asr_increase("/home2/liyang/GitHub/Mirage/Mirage/saved_logs/_Aug.06_22.18.27_fltrust/backdoor_tracking_log_nodefense.csv")

/tmp/ipykernel_4014955/812404567.py:87: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
